# Stage 2: Graph-Based Manga Recommendation Baselines

This notebook starts **Stage 2** of the CSE 694 project.

Goals for this notebook:
- load the existing SQLite dataset directly
- inspect the graph-relevant tables
- convert the database into graph-ready node and edge tables
- implement first-pass graph heuristics for recommendation
- keep the workflow **portable** and **independent of the Flask app**

This notebook is designed to run from a copy of the `/shelf` repo on your Raspberry Pi, laptop, or desktop.

## Portability Notes

This notebook intentionally avoids the web app and Flask service layer.

It only needs:
- the repo directory
- `data/db/manga.db`
- Python packages for data work (`pandas`, `numpy`, `scikit-learn` if you later compare to baseline logic)

Recommended setup on another machine:
1. Copy or clone the entire `/shelf` repo.
2. Make sure `data/db/manga.db` is present.
3. Open this notebook from the repo's `notebook/` folder.
4. Run the cells top to bottom.

In [ ]:
from pathlib import Path
import ast
import math
import sqlite3
from collections import Counter

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

NOTEBOOK_DIR = Path.cwd().resolve()
candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
candidate_roots.extend([NOTEBOOK_DIR / 'shelf', *[parent / 'shelf' for parent in NOTEBOOK_DIR.parents]])

seen = set()
REPO_ROOT = None
for candidate in candidate_roots:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if (candidate / 'data' / 'db' / 'manga.db').exists():
        REPO_ROOT = candidate
        break

assert REPO_ROOT is not None, 'Could not locate repo root containing data/db/manga.db'

DB_PATH = REPO_ROOT / 'data' / 'db' / 'manga.db'
EXAMPLE_USER_ID = 'avreylavelle'

print(f'Repo root: {REPO_ROOT}')
print(f'Database path: {DB_PATH}')
print(f'Database size (MB): {DB_PATH.stat().st_size / 1024**2:.1f}')
print(f'Example user for heuristic previews: {EXAMPLE_USER_ID}')


## Stage 2 Graph Schema

The working heterogeneous graph in this notebook uses:

**Node types**
- `user`
- `manga`
- `genre`
- `theme`
- `author`

**Edge types**
- `user -> rated -> manga`
- `user -> plans_to_read -> manga`
- `user -> in_progress -> manga`
- `user -> avoid -> manga`
- `manga -> has_genre -> genre`
- `manga -> has_theme -> theme`
- `manga -> written_by -> author`

This is enough to support Stage 2 graph heuristics such as:
- Common Neighbors
- Jaccard similarity
- Adamic-Adar

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    table_names = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY name",
        conn,
    )['name'].tolist()

    table_counts = []
    for name in table_names:
        count = conn.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0]
        table_counts.append({'table_name': name, 'row_count': count})

    table_counts_df = pd.DataFrame(table_counts).sort_values('row_count', ascending=False)

display(table_counts_df)

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    users_df = pd.read_sql_query('SELECT * FROM users', conn)
    ratings_df = pd.read_sql_query('SELECT * FROM user_ratings', conn)
    reading_df = pd.read_sql_query('SELECT * FROM user_reading_list', conn)
    dnr_df = pd.read_sql_query('SELECT * FROM user_dnr', conn)
    manga_df = pd.read_sql_query(
        '''
        SELECT
            core.id AS manga_id,
            core.title_name,
            core.english_name,
            core.japanese_name,
            core.item_type,
            core.authors,
            core.genres,
            core.themes,
            core.status,
            core.publishing_date,
            core.demographic,
            map.mal_id,
            stats.score,
            stats.popularity,
            stats.members,
            stats.favorited
        FROM manga_core AS core
        LEFT JOIN manga_map AS map ON map.mangadex_id = core.id
        LEFT JOIN manga_stats AS stats ON stats.mal_id = map.mal_id
        ''',
        conn,
    )

print('users_df:', users_df.shape)
print('ratings_df:', ratings_df.shape)
print('reading_df:', reading_df.shape)
print('dnr_df:', dnr_df.shape)
print('manga_df:', manga_df.shape)

In [ ]:
preview_frames = {
    'users': users_df.head(3),
    'ratings': ratings_df.head(3),
    'reading_list': reading_df.head(3),
    'dnr': dnr_df.head(3),
    'manga': manga_df.head(3),
}

for name, frame in preview_frames.items():
    print(f'\n{name}')
    display(frame)

In [ ]:
def parse_literal_list(value):
    """Parse stringified Python-list fields from SQLite into clean Python lists."""
    if value is None:
        return []
    if isinstance(value, list):
        return [
            str(item).strip()
            for item in value
            if str(item).strip() and str(item).strip().lower() != 'none'
        ]

    text = str(value).strip()
    if not text or text.lower() in {'none', 'nan'} or text == '[]':
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [
                str(item).strip()
                for item in parsed
                if str(item).strip() and str(item).strip().lower() != 'none'
            ]
    except Exception:
        pass

    return [part.strip() for part in text.split(',') if part.strip()]


def pick_manga_id(row):
    """Choose the most normalized manga identifier available in the interaction tables."""
    for column in ('canonical_id', 'mdex_id', 'manga_id'):
        value = row.get(column)
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return None


def explode_relation(frame, source_col, edge_type, target_type):
    """Explode manga -> metadata list fields into graph edge rows."""
    temp = frame[['manga_id', source_col]].copy()
    temp[source_col] = temp[source_col].apply(parse_literal_list)
    temp = temp.explode(source_col).dropna(subset=[source_col])
    temp[source_col] = temp[source_col].astype(str).str.strip()
    temp = temp[temp[source_col] != '']
    temp = temp.drop_duplicates(subset=['manga_id', source_col])
    return pd.DataFrame(
        {
            'source_id': temp['manga_id'],
            'source_type': 'manga',
            'target_id': temp[source_col],
            'target_type': target_type,
            'edge_type': edge_type,
            'edge_weight': 1.0,
        }
    )

In [ ]:
ratings_edges = ratings_df.copy()
ratings_edges['user_id'] = ratings_edges['user_id'].astype(str).str.strip().str.lower()
ratings_edges['manga_id'] = ratings_edges.apply(pick_manga_id, axis=1)
ratings_edges['rating'] = pd.to_numeric(ratings_edges['rating'], errors='coerce')
ratings_edges = ratings_edges.dropna(subset=['manga_id'])
ratings_edges_graph = pd.DataFrame(
    {
        'source_id': ratings_edges['user_id'],
        'source_type': 'user',
        'target_id': ratings_edges['manga_id'],
        'target_type': 'manga',
        'edge_type': 'rated',
        'edge_weight': ratings_edges['rating'].fillna(0.0),
        'rating': ratings_edges['rating'],
        'recommended_by_us': ratings_edges.get('recommended_by_us', 0),
        'finished_reading': ratings_edges.get('finished_reading', 0),
    }
)

reading_edges = reading_df.copy()
reading_edges['user_id'] = reading_edges['user_id'].astype(str).str.strip().str.lower()
reading_edges['manga_id'] = reading_edges.apply(pick_manga_id, axis=1)
reading_edges['status'] = reading_edges['status'].fillna('Plan to Read')
reading_edges = reading_edges.dropna(subset=['manga_id'])
reading_edges_graph = pd.DataFrame(
    {
        'source_id': reading_edges['user_id'],
        'source_type': 'user',
        'target_id': reading_edges['manga_id'],
        'target_type': 'manga',
        'edge_type': reading_edges['status'].str.lower().map(
            lambda value: 'in_progress' if value == 'in progress' else 'plans_to_read'
        ),
        'edge_weight': reading_edges['status'].str.lower().map(
            lambda value: 0.75 if value == 'in progress' else 0.5
        ),
        'status': reading_edges['status'],
    }
)

dnr_edges = dnr_df.copy()
dnr_edges['user_id'] = dnr_edges['user_id'].astype(str).str.strip().str.lower()
dnr_edges['manga_id'] = dnr_edges.apply(pick_manga_id, axis=1)
dnr_edges = dnr_edges.dropna(subset=['manga_id'])
dnr_edges_graph = pd.DataFrame(
    {
        'source_id': dnr_edges['user_id'],
        'source_type': 'user',
        'target_id': dnr_edges['manga_id'],
        'target_type': 'manga',
        'edge_type': 'avoid',
        'edge_weight': -1.0,
    }
)

genre_edges = explode_relation(manga_df, 'genres', 'has_genre', 'genre')
theme_edges = explode_relation(manga_df, 'themes', 'has_theme', 'theme')
author_edges = explode_relation(manga_df, 'authors', 'written_by', 'author')

user_manga_edges = pd.concat(
    [ratings_edges_graph, reading_edges_graph, dnr_edges_graph],
    ignore_index=True,
    sort=False,
)
metadata_edges = pd.concat(
    [genre_edges, theme_edges, author_edges],
    ignore_index=True,
    sort=False,
)
all_edges = pd.concat([user_manga_edges, metadata_edges], ignore_index=True, sort=False)

print('user_manga_edges:', user_manga_edges.shape)
print('metadata_edges:', metadata_edges.shape)
print('all_edges:', all_edges.shape)

In [ ]:
interaction_edge_summary = (
    user_manga_edges.groupby('edge_type')
    .size()
    .reset_index(name='edge_count')
    .sort_values('edge_count', ascending=False)
)

metadata_edge_summary = (
    metadata_edges.groupby('edge_type')
    .size()
    .reset_index(name='edge_count')
    .sort_values('edge_count', ascending=False)
)

print('User -> manga edges')
display(interaction_edge_summary)
print('\nManga -> metadata edges')
display(metadata_edge_summary)

In [ ]:
user_nodes = pd.DataFrame(
    {
        'node_id': users_df['username'].astype(str).str.strip().str.lower(),
        'node_type': 'user',
    }
).drop_duplicates()

manga_nodes = manga_df[
    ['manga_id', 'title_name', 'english_name', 'item_type', 'status', 'publishing_date', 'mal_id', 'score']
].drop_duplicates(subset=['manga_id']).rename(columns={'manga_id': 'node_id'})
manga_nodes['node_type'] = 'manga'

genre_nodes = pd.DataFrame({'node_id': sorted(genre_edges['target_id'].unique()), 'node_type': 'genre'})
theme_nodes = pd.DataFrame({'node_id': sorted(theme_edges['target_id'].unique()), 'node_type': 'theme'})
author_nodes = pd.DataFrame({'node_id': sorted(author_edges['target_id'].unique()), 'node_type': 'author'})

node_tables = {
    'user': user_nodes,
    'manga': manga_nodes,
    'genre': genre_nodes,
    'theme': theme_nodes,
    'author': author_nodes,
}

node_summary = pd.DataFrame(
    [{'node_type': node_type, 'node_count': len(frame)} for node_type, frame in node_tables.items()]
).sort_values('node_count', ascending=False)

display(node_summary)

In [ ]:
graph_schema_summary = pd.DataFrame(
    [
        {'edge_type': 'rated', 'source_type': 'user', 'target_type': 'manga', 'edge_count': len(ratings_edges_graph)},
        {'edge_type': 'plans_to_read', 'source_type': 'user', 'target_type': 'manga', 'edge_count': int((reading_edges_graph['edge_type'] == 'plans_to_read').sum())},
        {'edge_type': 'in_progress', 'source_type': 'user', 'target_type': 'manga', 'edge_count': int((reading_edges_graph['edge_type'] == 'in_progress').sum())},
        {'edge_type': 'avoid', 'source_type': 'user', 'target_type': 'manga', 'edge_count': len(dnr_edges_graph)},
        {'edge_type': 'has_genre', 'source_type': 'manga', 'target_type': 'genre', 'edge_count': len(genre_edges)},
        {'edge_type': 'has_theme', 'source_type': 'manga', 'target_type': 'theme', 'edge_count': len(theme_edges)},
        {'edge_type': 'written_by', 'source_type': 'manga', 'target_type': 'author', 'edge_count': len(author_edges)},
    ]
).sort_values('edge_count', ascending=False)

display(graph_schema_summary)

## Stage 2 Heuristic Baselines

The next step is to use the heterogeneous graph for simple recommendation heuristics before jumping to learned graph models.

This notebook starts with a pragmatic interpretation of the graph:
- build a user profile from strong positive user -> manga edges
- compare candidate manga against that profile through shared metadata neighbors
- rank candidates using three classical heuristics:
  - **Common Neighbors**
  - **Jaccard similarity**
  - **Adamic-Adar**

Current assumptions for a first pass:
- strong positives = ratings >= 7 plus `In Progress` reading-list items
- blocked items = anything already rated, in reading list, or in DNR
- metadata relations used = genres, themes, authors

In [ ]:
manga_features = manga_df.copy()
for column in ('genres', 'themes', 'authors'):
    manga_features[column] = manga_features[column].apply(parse_literal_list)
    manga_features[f'{column}_set'] = manga_features[column].apply(set)

manga_features = manga_features.drop_duplicates(subset=['manga_id']).set_index('manga_id', drop=False)

relation_degrees = {}
for relation in ('genres', 'themes', 'authors'):
    counter = Counter()
    for tags in manga_features[relation]:
        counter.update(set(tags))
    relation_degrees[relation] = dict(counter)


def build_user_seed_items(user_id, rating_threshold=7.0):
    user_key = str(user_id).strip().lower()
    user_ratings = ratings_edges[ratings_edges['user_id'] == user_key]
    user_reading = reading_edges[reading_edges['user_id'] == user_key]
    user_dnr = dnr_edges[dnr_edges['user_id'] == user_key]

    strong_ratings = set(user_ratings.loc[user_ratings['rating'] >= rating_threshold, 'manga_id'])
    in_progress = set(user_reading.loc[user_reading['status'].str.lower() == 'in progress', 'manga_id'])

    seed_items = strong_ratings | in_progress
    blocked_items = set(user_ratings['manga_id']) | set(user_reading['manga_id']) | set(user_dnr['manga_id'])
    return seed_items, blocked_items


def build_user_metadata_profile(user_id, rating_threshold=7.0):
    seed_items, blocked_items = build_user_seed_items(user_id, rating_threshold=rating_threshold)
    profile = {relation: Counter() for relation in ('genres', 'themes', 'authors')}

    for manga_id in seed_items:
        if manga_id not in manga_features.index:
            continue
        row = manga_features.loc[manga_id]
        for relation in profile:
            profile[relation].update(row[relation])

    return seed_items, blocked_items, profile

In [ ]:
def stage2_recommend(user_id, method='common_neighbors', top_k=20, rating_threshold=7.0, relation_weights=None):
    """Recommend manga with a simple graph heuristic over user -> manga -> metadata -> manga paths."""
    relation_weights = relation_weights or {'genres': 1.0, 'themes': 1.0, 'authors': 2.0}
    seed_items, blocked_items, profile = build_user_metadata_profile(user_id, rating_threshold=rating_threshold)
    if not seed_items:
        return pd.DataFrame(columns=['manga_id', 'title_name', 'heuristic_score', 'matches'])

    profile_sets = {relation: set(counter.keys()) for relation, counter in profile.items()}
    rows = []

    for row in manga_features.itertuples(index=False):
        manga_id = row.manga_id
        if manga_id in blocked_items:
            continue

        total_score = 0.0
        matched = []
        for relation, relation_weight in relation_weights.items():
            candidate_tags = getattr(row, f'{relation}_set')
            shared = candidate_tags & profile_sets[relation]
            if not shared:
                continue

            if method == 'common_neighbors':
                relation_score = sum(profile[relation][tag] for tag in shared)
            elif method == 'jaccard':
                union = candidate_tags | profile_sets[relation]
                relation_score = len(shared) / len(union) if union else 0.0
            elif method == 'adamic_adar':
                relation_score = sum(
                    profile[relation][tag] / math.log1p(max(relation_degrees[relation].get(tag, 1), 1))
                    for tag in shared
                )
            else:
                raise ValueError(f'Unknown method: {method}')

            total_score += relation_weight * relation_score
            matched.extend(f"{relation[:-1]}:{tag}" for tag in sorted(shared)[:2])

        if total_score <= 0:
            continue

        rows.append(
            {
                'manga_id': manga_id,
                'title_name': row.title_name,
                'english_name': row.english_name,
                'item_type': row.item_type,
                'baseline_score': row.score,
                'heuristic_score': total_score,
                'matches': ', '.join(dict.fromkeys(matched)),
            }
        )

    results = pd.DataFrame(rows)
    if results.empty:
        return results

    results = results.sort_values(['heuristic_score', 'baseline_score'], ascending=[False, False])
    return results.head(top_k).reset_index(drop=True)

In [ ]:
seed_items, blocked_items, profile = build_user_metadata_profile(EXAMPLE_USER_ID)

print('Seed item count:', len(seed_items))
print('Blocked item count:', len(blocked_items))
print('Top profile genres:', profile['genres'].most_common(10))
print('Top profile themes:', profile['themes'].most_common(10))
print('Top profile authors:', profile['authors'].most_common(10))

seed_preview = (
    manga_features.loc[list(seed_items), ['title_name', 'english_name', 'item_type']]
    .sort_values('title_name')
    if seed_items
    else pd.DataFrame()
)
display(seed_preview.head(20))

In [ ]:
for method in ('common_neighbors', 'jaccard', 'adamic_adar'):
    print(f'\n### {method}')
    display(stage2_recommend(EXAMPLE_USER_ID, method=method, top_k=10))

## Good Next Steps For This Notebook

1. Build a **train/test split** over user -> manga edges.
2. Evaluate the Stage 2 heuristics using **Precision@K**, **Recall@K**, and **NDCG**.
3. Add a clean **baseline comparison cell** against the current content-based recommender.
4. Try weighting different relation types differently:
   - authors > themes > genres
   - or tune by validation results
5. Decide whether the Stage 3 model should use:
   - a projected graph
   - a heterogeneous graph object
   - or a PyTorch Geometric / DGL pipeline

This notebook should stay focused on experiment logic, not web-app integration.